# NIMBO Image Quality + Real Auto-Fallback — T4

**Phase 6:** Evaluate CLIP checker on the **existing actual images**, then run an end-to-end `LCM → CLIP → Standard if failed` experiment. Scores are **relative CLIP signals, not object-detector proof**. Do not tune thresholds using the evaluation set. Compare with always-standard including checker overhead; separate human review is needed. Requires a T4 GPU.

The previous results folder `nimbo_image_results` must exist or be uploaded into this Colab session for the first evaluation cell. The live pipeline runs regardless.

In [ ]:
%pip -q install "diffusers>=0.30,<1" "transformers>=4.51,<5" "accelerate>=0.30" "peft>=0.12" "safetensors>=0.4" pandas pillow
# If the previous torchao error recurs: %pip uninstall -y torchao ; restart session ; run again.

In [ ]:
import gc, json, time
from pathlib import Path
import torch, pandas as pd
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from diffusers import StableDiffusionPipeline, LCMScheduler
assert torch.cuda.is_available(), 'Enable T4 in Runtime settings.'
print('GPU',torch.cuda.get_device_name(0))
BASE='stable-diffusion-v1-5/stable-diffusion-v1-5'
LORA='latent-consistency/lcm-lora-sdv1-5'
OUTPUT=Path('nimbo_image_results'); OUTPUT.mkdir(exist_ok=True)
DEVICE='cuda'
PROMPTS=[
 ('a sunlit modern library with oak bookshelves, architectural photograph','books'),
 ('a small red sailboat on a calm lake at dawn, cinematic photography','sailboat'),
 ('a minimal white ceramic teapot on a wooden table, studio product photograph','teapot'),
 ('a futuristic clean energy city at sunset, wide angle photography','buildings'),
]
SEEDS=[17,23,41,59]


## A. Real CLIP diagnostics on images you have already generated

This cell **does not classify images as truth**. It prints raw scores so you can label which comparisons actually meet your prompt. Contrastive scores can be very confident and wrong. Initial threshold is only a provisional engineering setting.

In [ ]:
clip=CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
processor=CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def contrast(image, captions):
    inputs=processor(text=captions,images=image,return_tensors='pt',padding=True,truncation=True).to(DEVICE)
    with torch.inference_mode():
        torch.cuda.synchronize();t=time.perf_counter()
        scores=clip(**inputs).logits_per_image.softmax(-1)[0].cpu().tolist()
        torch.cuda.synchronize(); elapsed=time.perf_counter()-t
    return scores[0], elapsed

def assess(image,prompt,required_object,threshold=0.65):
    t=time.perf_counter()
    alignment,_=contrast(image,[prompt,'an unrelated abstract pattern'])
    presence,_=contrast(image,[f'a photograph showing a clearly visible {required_object}',f'a photograph with no {required_object} visible'])
    torch.cuda.synchronize()
    return {'alignment':round(alignment,4),'object_relative_score':round(presence,4),'passed':alignment>=threshold and presence>=threshold,'checker_s':round(time.perf_counter()-t,4)}

records=[]
for i,(prompt,obj) in enumerate(PROMPTS):
  for mode in ('standard','lcm'):
    path=OUTPUT/f'{mode}_prompt{i:02d}_run00.png'
    if path.exists():
      r=assess(Image.open(path).convert('RGB'),prompt,obj)
      records.append({'mode':mode,'prompt_id':i,'required_object':obj,**r})
if records:
  diag=pd.DataFrame(records)
  display(diag)
  diag.to_csv(OUTPUT/'clip_diagnostics.csv',index=False)
else:
  print('Previous PNGs not found. Upload nimbo_image_results to this Colab Files tab, or continue with the live test below.')

## B. Live auto-fallback (actual generation)

Uses one prompt at a time, seeds fixed. Baseline from prior notebook must **not** be compared with an end-to-end time measured under a different runtime without caveats. This cell times the full chain with CLIP and records each decision. Standard pipeline loads lazily for failed requests (model loading itself is **excluded** from inference latency, but separately recorded). For a production SLA it must be included or the model must be kept warm.

**Important:** `safety_checker=None` follows the prior research notebook and is **not appropriate for a public-facing image generation service**. Restore appropriate safety moderation before any public deployment.

In [ ]:
def build_pipe(mode):
    pipe=StableDiffusionPipeline.from_pretrained(BASE,torch_dtype=torch.float16,safety_checker=None).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    if mode=='lcm':
        pipe.load_lora_weights(LORA)
        pipe.scheduler=LCMScheduler.from_config(pipe.scheduler.config)
    return pipe

def render(pipe,mode,prompt,seed):
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); start=time.perf_counter()
    with torch.inference_mode():
        image=pipe(prompt,num_inference_steps=4 if mode=='lcm' else 25,
                   guidance_scale=1.0 if mode=='lcm' else 7.5,
                   generator=torch.Generator(device='cuda').manual_seed(seed),
                   width=512,height=512).images[0]
    torch.cuda.synchronize()
    return image,time.perf_counter()-start,torch.cuda.max_memory_allocated()/1e9

# CLIP occupies GPU memory; load generators one at a time to fit T4.
# A larger GPU can keep both resident for low fallback latency.
fast=build_pipe('lcm')
_ = render(fast,'lcm',PROMPTS[0][0],SEEDS[0])  # excluded warmup
live=[]
for i,(prompt,obj) in enumerate(PROMPTS):
    image,fast_s,fast_gb=render(fast,'lcm',prompt,SEEDS[i])
    check=assess(image,prompt,obj,threshold=0.65)
    image.save(OUTPUT/f'auto_fast_{i:02d}.png')
    live.append({'prompt_id':i,'prompt':prompt,'object':obj,'fast_s':round(fast_s,4),
                 'fast_peak_allocated_gb':round(fast_gb,3),**check,
                 'route':'fast_accepted' if check['passed'] else 'needs_standard',
                 'final_image':f'auto_fast_{i:02d}.png'})
    print(i,live[-1]['route'],check)
del fast; gc.collect();torch.cuda.empty_cache()


In [ ]:
# Only generate standard on failed checks. Time actual GPU inference separately
# and the model-loading overhead separately. Do not count standard latency for passes.
failed=[r for r in live if r['route']=='needs_standard']
load_s=0.0
if failed:
    start=time.perf_counter()
    standard=build_pipe('standard')
    load_s=time.perf_counter()-start
    _=render(standard,'standard',PROMPTS[0][0],SEEDS[0])
    for r in failed:
        i=r['prompt_id']; img,elapsed,peak=render(standard,'standard',PROMPTS[i][0],SEEDS[i])
        img.save(OUTPUT/f'auto_standard_{i:02d}.png')
        r.update({'standard_s':round(elapsed,4),'standard_peak_allocated_gb':round(peak,3),
                  'route':'escalated_to_standard','final_image':f'auto_standard_{i:02d}.png'})
    del standard;gc.collect();torch.cuda.empty_cache()
for r in live:
    r['standard_s']=r.get('standard_s',0.0)
    r['end_to_end_inference_s']=round(r['fast_s']+r['checker_s']+r['standard_s'],4)
report={'gpu':torch.cuda.get_device_name(0),'threshold_provisional':0.65,
        'prompt_count':len(live),'fallback_count':len(failed),
        'avg_end_to_end_inference_s':round(sum(r['end_to_end_inference_s'] for r in live)/len(live),4),
        'standard_loading_s_if_needed':round(load_s,3),
        'model_loading_included_in_latency':False,
        'quality_validated':False,
        'warning':'CLIP relative scores are not calibrated and cannot establish object presence or image quality; human labeling required.',
        'results':live}
(OUTPUT/'quality_auto_routing.json').write_text(json.dumps(report,indent=2))
pd.DataFrame(live).to_csv(OUTPUT/'quality_auto_routing.csv',index=False)
print(json.dumps({k:v for k,v in report.items() if k!='results'},indent=2))
display(pd.DataFrame(live)[['prompt_id','route','alignment','object_relative_score','checker_s','end_to_end_inference_s']])


## C. Human verification (mandatory before quality claims)

Open `auto_fast_*.png` and `auto_standard_*.png` (where escalated). For each prompt label `critical_object_visible = 0/1`, and `prompt_followed = 0/1`. Check whether CLIP's `passed` flag agrees; calculate false pass rate. **Do not change threshold based on these 4 evaluation examples**. Make a separate, larger labeled calibration set before choosing threshold. In the notebook Files panel, zip and download the result folder.


In [ ]:
# Enter independent human observations, 1=present, 0=missing, None=unreviewed
human_labels={0:None,1:None,2:None,3:None}
review=[{'prompt_id':r['prompt_id'],'clip_passed':r['passed'],'human_critical_object_present':human_labels[r['prompt_id']]} for r in live]
display(pd.DataFrame(review))
(OUTPUT/'human_review_template.json').write_text(json.dumps(review,indent=2))
print('Download with: !zip -qr nimbo_image_results_phase6.zip nimbo_image_results')